<div class="lesson-banner">
<span class="lesson-kicker">Python course · 2-hour lesson</span>
<p>Make failures visible and actionable with precise exceptions, systematic debugging, and structured logs.</p>
</div>

## Learning objectives

- Distinguish syntax errors, exceptions, and incorrect results.
- Catch only errors a layer can handle.
- Create informative domain exceptions.
- Use assertions, breakpoints, and logging for diagnosis.

::: {.callout-note}
### How to use this notebook
Read the explanation, predict each result, run the code, change the inputs, and complete the practice before revealing the solution.
:::


## Exception boundaries

An exception should cross layers until a layer can recover, add context, or present a user-facing message. Catch specific exception types; a broad `except Exception` can hide programming mistakes. Use `else` for code that runs only after success and `finally` for cleanup that must always occur.


In [ ]:
def parse_port(raw: str) -> int:
    try:
        port = int(raw)
    except ValueError as error:
        raise ValueError(f"port must be an integer, received {raw!r}") from error
    if not 1 <= port <= 65_535:
        raise ValueError("port must be between 1 and 65535")
    return port


for candidate in ["8000", "abc", "70000"]:
    try:
        print(candidate, parse_port(candidate))
    except ValueError as error:
        print("Invalid configuration:", error)


## Debug from evidence

Reproduce the smallest failing case, inspect inputs and intermediate values, state a hypothesis, then test it. `repr`, type checks, assertions, and a debugger reveal state without random edits. Assertions document internal invariants; do not use them to validate untrusted user input because optimized Python can disable them.


In [ ]:
def normalized_average(values: list[float]) -> float:
    assert isinstance(values, list), "internal contract expects a list"
    if not values:
        raise ValueError("values cannot be empty")
    total = sum(values)
    count = len(values)
    assert count > 0
    return total / count


sample = [10.0, 20.0, 40.0]
print({"input": repr(sample), "result": normalized_average(sample)})


## Logging records operational context

Logs answer what happened, where, and with what safe context. Use log levels consistently: DEBUG for diagnostic detail, INFO for normal milestones, WARNING for recoverable concern, ERROR for failed work, and CRITICAL for service-threatening failures. Never log passwords, tokens, or sensitive personal data.


In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s %(name)s %(message)s",
)
logger = logging.getLogger("course.pipeline")

records = [{"id": 1, "score": 82}, {"id": 2, "score": None}]
for record in records:
    if record["score"] is None:
        logger.warning("record skipped id=%s reason=missing_score", record["id"])
        continue
    logger.info("record accepted id=%s", record["id"])


## Worked example: collect row-level validation errors

Batch processing often should continue after a bad row while preserving a complete error report.


In [ ]:
def validate_record(record: dict) -> dict:
    try:
        learner_id = int(record["id"])
        score = float(record["score"])
    except KeyError as error:
        raise ValueError(f"missing field: {error.args[0]}") from error
    except (TypeError, ValueError) as error:
        raise ValueError("id and score must be numeric") from error
    if not 0 <= score <= 100:
        raise ValueError("score outside 0..100")
    return {"id": learner_id, "score": score}


raw_records = [{"id": "1", "score": "88"}, {"id": "x", "score": "72"}]
valid, errors = [], []
for position, record in enumerate(raw_records, start=1):
    try:
        valid.append(validate_record(record))
    except ValueError as error:
        errors.append({"row": position, "error": str(error)})
print({"valid": valid, "errors": errors})


## Practice lab

Complete these tasks without copying the solution. Test normal, boundary, and invalid inputs where relevant.

1. Write a parser for percentages such as `82.5%` with clear error messages.
2. Process five records, collecting every validation error instead of stopping at the first.
3. Add INFO logs for start/end and WARNING logs for rejected records.
4. Diagnose an intentionally wrong average by printing the smallest useful intermediate state.

::: {.callout-important}
### Practice standard
Your answer should be readable, deterministic, and divided into small functions when the task contains more than one rule.
:::


## Suggested solution

Open the folded code only after attempting every task.


In [ ]:
import logging

logger = logging.getLogger("percentage-import")


def parse_percentage(raw: str) -> float:
    if not isinstance(raw, str) or not raw.endswith("%"):
        raise ValueError("percentage must be text ending with %")
    try:
        value = float(raw[:-1])
    except ValueError as error:
        raise ValueError(f"invalid percentage: {raw!r}") from error
    if not 0 <= value <= 100:
        raise ValueError("percentage must be between 0% and 100%")
    return value / 100


values = ["82.5%", "100%", "bad", "120%"]
accepted, rejected = [], []
logger.info("import started count=%s", len(values))
for value in values:
    try:
        accepted.append(parse_percentage(value))
    except ValueError as error:
        rejected.append({"value": value, "error": str(error)})
        logger.warning("value rejected value=%r", value)
logger.info("import finished accepted=%s rejected=%s", len(accepted), len(rejected))
print(accepted, rejected)


## Knowledge check

**1. When should you catch an exception?**

::: {.callout-note collapse="true"}
### Answer
When the current layer can recover, add useful context, or present it appropriately.
:::

**2. Are assertions input validation?**

::: {.callout-note collapse="true"}
### Answer
No; use explicit exceptions for external input.
:::

**3. What must never enter logs?**

::: {.callout-note collapse="true"}
### Answer
Secrets and sensitive personal data.
:::


## Recap

- Catch narrowly.
- Debug from a reproducible case and observed state.
- Write logs for operators, not for decoration.


<div class="lesson-nav">
<a href="06-files-and-data-formats.html"><i class="bi bi-arrow-left" aria-hidden="true"></i> Files, Paths, JSON, and CSV</a>
<a href="08-modules-packages-environments.html">Modules, Packages, and Environments <i class="bi bi-arrow-right" aria-hidden="true"></i></a>
</div>
